In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter

# -----------------------------
# DATA
# -----------------------------
data = {
    "Pass":         {"blue_success": 147, "blue_fail": 68, "red_success": 207, "red_fail": 94},
    "Interception": {"blue_success": 24,  "blue_fail": 1,  "red_success": 22,  "red_fail": 2},
    "Dribble":      {"blue_success": 10,  "blue_fail": 4,  "red_success": 14,  "red_fail": 8},
    "Cross":        {"blue_success": 3,   "blue_fail": 17, "red_success": 3,   "red_fail": 16},
    "Throw In":     {"blue_success": 10,  "blue_fail": 14, "red_success": 17,  "red_fail": 11},
    "Tackle":       {"blue_success": 21,  "blue_fail": 0,  "red_success": 17,  "red_fail": 2},
    "Corner":       {"blue_success": 6,   "blue_fail": 1,  "red_success": 2,   "red_fail": 4},
    "Free Kick":    {"blue_success": 1,   "blue_fail": 9,  "red_success": 7,   "red_fail": 3},
}

counts_only = {
    "Clearances": {"blue": 63, "red": 45},
    "Fouls":      {"blue": 7,  "red": 15},
    "Offsides":   {"blue": 4,  "red": 1},
}

shots = {
    "blue_on": 8,
    "blue_off": 4,
    "red_on": 5,
    "red_off": 8,
}

# -----------------------------
# STYLE
# -----------------------------
BG = "#071221"
TEXT = "#EAF0FA"
MUTED = "#A6B5CC"
GRID = "#294463"
BLUE = "#59A8FF"
BLUE_SOFT = "#A9D0FF"
RED = "#FF7A7A"
RED_SOFT = "#FFC0C0"
TRACK = "#50627A"

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.facecolor"] = "none"
plt.rcParams["figure.facecolor"] = "none"

# -----------------------------
# HELPER DF
# -----------------------------
rows = []
for metric, vals in data.items():
    bt = vals["blue_success"] + vals["blue_fail"]
    rt = vals["red_success"] + vals["red_fail"]
    rows.append({
        "Metric": metric,
        "Blue Success": vals["blue_success"],
        "Blue Fail": vals["blue_fail"],
        "Blue Total": bt,
        "Blue Rate": vals["blue_success"] / bt if bt else np.nan,
        "Red Success": vals["red_success"],
        "Red Fail": vals["red_fail"],
        "Red Total": rt,
        "Red Rate": vals["red_success"] / rt if rt else np.nan,
    })
rates_df = pd.DataFrame(rows)

# -----------------------------
# 1) VOLUME BALANCE
# -----------------------------
fig, ax = plt.subplots(figsize=(8, 4.8), dpi=300)

metrics = ["Passes", "Clearances", "Fouls", "Offsides"]
blue_vals = np.array([
    -(data["Pass"]["blue_success"] + data["Pass"]["blue_fail"]),
    -counts_only["Clearances"]["blue"],
    -counts_only["Fouls"]["blue"],
    -counts_only["Offsides"]["blue"],
])
red_vals = np.array([
    data["Pass"]["red_success"] + data["Pass"]["red_fail"],
    counts_only["Clearances"]["red"],
    counts_only["Fouls"]["red"],
    counts_only["Offsides"]["red"],
])

y = np.arange(len(metrics))
ax.barh(y, blue_vals, height=0.48, color=BLUE, alpha=0.92)
ax.barh(y, red_vals, height=0.48, color=RED, alpha=0.88)
ax.axvline(0, color=MUTED, lw=1)

ax.set_yticks(y)
ax.set_yticklabels(metrics, color=TEXT, fontsize=11)
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{abs(int(x))}"))
ax.tick_params(axis="x", colors=MUTED, labelsize=9)
ax.grid(axis="x", color=GRID, linestyle="--", alpha=0.25)
ax.set_xlim(-330, 330)
ax.invert_yaxis()

for i, v in enumerate(blue_vals):
    ax.text(v - 7, i, f"{abs(int(v))}", ha="right", va="center", color=TEXT, fontsize=10)
for i, v in enumerate(red_vals):
    ax.text(v + 7, i, f"{int(v)}", ha="left", va="center", color=TEXT, fontsize=10)

for s in ax.spines.values():
    s.set_visible(False)

ax.text(0.02, 1.04, "Blue", transform=ax.transAxes, color=BLUE_SOFT, fontsize=10, fontweight="bold")
ax.text(0.98, 1.04, "Red", transform=ax.transAxes, color=RED_SOFT, fontsize=10, fontweight="bold", ha="right")

plt.tight_layout()
plt.savefig("01_volume_balance.png", transparent=True, bbox_inches="tight", pad_inches=0.15)
plt.close()

# -----------------------------
# 2) SUCCESS RATES
# -----------------------------
fig, ax = plt.subplots(figsize=(7.6, 4.8), dpi=300)

compare_metrics = ["Pass", "Interception", "Dribble", "Tackle"]
plot_df = rates_df[rates_df["Metric"].isin(compare_metrics)].copy()
plot_df["Blue %"] = plot_df["Blue Rate"] * 100
plot_df["Red %"] = plot_df["Red Rate"] * 100
plot_df = plot_df.sort_values("Blue %", ascending=True).reset_index(drop=True)

y = np.arange(len(plot_df))
ax.hlines(y, plot_df["Red %"], plot_df["Blue %"], color=TRACK, lw=2.2, alpha=0.9)
ax.scatter(plot_df["Red %"], y, s=90, color=RED, zorder=3)
ax.scatter(plot_df["Blue %"], y, s=90, color=BLUE, zorder=3)

ax.set_yticks(y)
ax.set_yticklabels(plot_df["Metric"], color=TEXT, fontsize=11)
ax.set_xlim(0, 105)
ax.xaxis.set_major_formatter(PercentFormatter(xmax=100))
ax.tick_params(axis="x", colors=MUTED, labelsize=9)
ax.grid(axis="x", color=GRID, linestyle="--", alpha=0.25)

for i, row in plot_df.iterrows():
    ax.text(row["Blue %"] + 2, i, f'{row["Blue %"]:.1f}%', color=BLUE_SOFT, fontsize=9, va="center")
    ax.text(row["Red %"] - 2, i, f'{row["Red %"]:.1f}%', color=RED_SOFT, fontsize=9, va="center", ha="right")

for s in ax.spines.values():
    s.set_visible(False)

plt.tight_layout()
plt.savefig("02_success_rates.png", transparent=True, bbox_inches="tight", pad_inches=0.15)
plt.close()

# -----------------------------
# 3) RESTART RETENTION
# -----------------------------
fig, ax = plt.subplots(figsize=(8.2, 4.8), dpi=300)

restart_df = pd.DataFrame({
    "Metric": ["Free Kick", "Cross", "Throw In", "Corner"],
    "Blue %": [
        rates_df.loc[rates_df["Metric"] == "Free Kick", "Blue Rate"].iloc[0] * 100,
        rates_df.loc[rates_df["Metric"] == "Cross", "Blue Rate"].iloc[0] * 100,
        rates_df.loc[rates_df["Metric"] == "Throw In", "Blue Rate"].iloc[0] * 100,
        rates_df.loc[rates_df["Metric"] == "Corner", "Blue Rate"].iloc[0] * 100,
    ],
    "Red %": [
        rates_df.loc[rates_df["Metric"] == "Free Kick", "Red Rate"].iloc[0] * 100,
        rates_df.loc[rates_df["Metric"] == "Cross", "Red Rate"].iloc[0] * 100,
        rates_df.loc[rates_df["Metric"] == "Throw In", "Red Rate"].iloc[0] * 100,
        rates_df.loc[rates_df["Metric"] == "Corner", "Red Rate"].iloc[0] * 100,
    ]
})

y = np.arange(len(restart_df))
ax.hlines(y, 0, restart_df["Blue %"], color=BLUE_SOFT, lw=6, alpha=0.35)
ax.scatter(restart_df["Blue %"], y, s=150, color=BLUE, zorder=3, label="Blue")
ax.scatter(restart_df["Red %"], y, s=200, color=RED, marker="|", linewidths=3, zorder=4, label="Red benchmark")

ax.set_yticks(y)
ax.set_yticklabels(restart_df["Metric"], color=TEXT, fontsize=11)
ax.set_xlim(0, 100)
ax.xaxis.set_major_formatter(PercentFormatter(xmax=100))
ax.tick_params(axis="x", colors=MUTED, labelsize=9)
ax.grid(axis="x", color=GRID, linestyle="--", alpha=0.25)

for i, row in restart_df.iterrows():
    ax.text(row["Blue %"] + 2, i, f'{row["Blue %"]:.1f}%', color=BLUE_SOFT, fontsize=10, va="center")

for s in ax.spines.values():
    s.set_visible(False)

plt.tight_layout()
plt.savefig("03_restart_retention.png", transparent=True, bbox_inches="tight", pad_inches=0.15)
plt.close()

# -----------------------------
# 4) SHOT PROFILE
# -----------------------------
fig = plt.figure(figsize=(7.6, 4.8), dpi=300)
ax1 = fig.add_axes([0.02, 0.05, 0.43, 0.9])
ax2 = fig.add_axes([0.55, 0.05, 0.43, 0.9])

for ax, team, on_val, off_val, color in [
    (ax1, "Blue", shots["blue_on"], shots["blue_off"], BLUE),
    (ax2, "Red", shots["red_on"], shots["red_off"], RED),
]:
    total = on_val + off_val
    pct = on_val / total
    ax.pie(
        [on_val, off_val],
        startangle=90,
        counterclock=False,
        colors=[color, "#334760"],
        wedgeprops=dict(width=0.32, edgecolor="none")
    )
    ax.text(0, 0.12, team, ha="center", color=TEXT, fontsize=13, fontweight="bold")
    ax.text(0, -0.04, f"{pct*100:.1f}%", ha="center", color=TEXT, fontsize=22, fontweight="bold")
    ax.text(0, -0.22, "on target", ha="center", color=MUTED, fontsize=10)
    ax.text(0, -0.45, f"{total} shots", ha="center", color=MUTED, fontsize=10)
    ax.set_aspect("equal")
    ax.axis("off")

plt.savefig("04_shot_profile.png", transparent=True, bbox_inches="tight", pad_inches=0.15)
plt.close()

print("Saved:")
print("01_volume_balance.png")
print("02_success_rates.png")
print("03_restart_retention.png")
print("04_shot_profile.png")

Saved:
01_volume_balance.png
02_success_rates.png
03_restart_retention.png
04_shot_profile.png


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from matplotlib.patches import FancyBboxPatch, Rectangle

# =========================
# DATA
# =========================
blue = {
    "pass_pct": 147/215 * 100,
    "interception_pct": 24/25 * 100,
    "dribble_pct": 10/14 * 100,
    "tackle_pct": 21/21 * 100,
    "corner_pct": 6/7 * 100,
    "throwin_pct": 10/24 * 100,
    "cross_pct": 3/20 * 100,
    "freekick_pct": 1/10 * 100,
    "shots_on": 8,
    "shots_total": 12,
}

red = {
    "pass_pct": 207/301 * 100,
    "interception_pct": 22/24 * 100,
    "dribble_pct": 14/22 * 100,
    "tackle_pct": 17/19 * 100,
    "corner_pct": 2/6 * 100,
    "throwin_pct": 17/28 * 100,
    "cross_pct": 3/19 * 100,
    "freekick_pct": 7/10 * 100,
    "shots_on": 5,
    "shots_total": 13,
}

# =========================
# STYLE
# =========================
TEXT = "#EAF0FA"
MUTED = "#A6B5CC"
GRID = "#2A4362"
BLUE = "#59A8FF"
BLUE_SOFT = "#A9D0FF"
RED = "#FF7A7A"
RED_SOFT = "#FFC2C2"
TRACK = "#334760"
CARD_BG = "#0D1B2E"
CARD_EDGE = "#213652"
GOOD = "#32D7A6"
BAD = "#FF7A7A"
NEUTRAL = "#F6C85F"

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.facecolor"] = "none"
plt.rcParams["figure.facecolor"] = "none"

# =========================
# 1) SHOT PROFILE: 100% STACKED BARS
# =========================
fig, ax = plt.subplots(figsize=(8.2, 3.8), dpi=300)

teams = ["Blue", "Red"]
on_pct = [
    blue["shots_on"] / blue["shots_total"] * 100,
    red["shots_on"] / red["shots_total"] * 100,
]
off_pct = [100 - p for p in on_pct]
totals = [blue["shots_total"], red["shots_total"]]
on_counts = [blue["shots_on"], red["shots_on"]]
colors = [BLUE, RED]

y = np.arange(len(teams))

ax.barh(y, on_pct, color=colors, height=0.36)
ax.barh(y, off_pct, left=on_pct, color=TRACK, height=0.36)

ax.set_xlim(0, 100)
ax.set_yticks(y)
ax.set_yticklabels(teams, color=TEXT, fontsize=12, fontweight="bold")
ax.xaxis.set_major_formatter(PercentFormatter())
ax.tick_params(axis="x", colors=MUTED, labelsize=9)
ax.grid(axis="x", color=GRID, linestyle="--", alpha=0.25)

for i, pct in enumerate(on_pct):
    ax.text(1.5, i, f"{pct:.1f}%", va="center", ha="left",
            color=TEXT, fontsize=14, fontweight="bold")
    ax.text(98.5, i, f"{on_counts[i]}/{totals[i]} on target", va="center", ha="right",
            color=MUTED, fontsize=10)

for s in ax.spines.values():
    s.set_visible(False)

ax.invert_yaxis()
plt.tight_layout()
plt.savefig("shot_profile_bars.png", transparent=True, bbox_inches="tight", pad_inches=0.15)
plt.close()

# =========================
# 2) SUCCESS DELTA: BLUE MINUS RED
# =========================
fig, ax = plt.subplots(figsize=(7.6, 4.2), dpi=300)

metrics = ["Tackle", "Dribble", "Interception", "Pass"]
delta = np.array([
    blue["tackle_pct"] - red["tackle_pct"],
    blue["dribble_pct"] - red["dribble_pct"],
    blue["interception_pct"] - red["interception_pct"],
    blue["pass_pct"] - red["pass_pct"],
])

colors = [BLUE if d >= 0 else RED for d in delta]
y = np.arange(len(metrics))

ax.barh(y, delta, color=colors, height=0.48)
ax.axvline(0, color=MUTED, lw=1.2)

ax.set_yticks(y)
ax.set_yticklabels(metrics, color=TEXT, fontsize=12)
ax.set_xlim(-15, 15)
ax.tick_params(axis="x", colors=MUTED, labelsize=9)
ax.grid(axis="x", color=GRID, linestyle="--", alpha=0.25)

for i, d in enumerate(delta):
    ha = "left" if d >= 0 else "right"
    x = d + 0.5 if d >= 0 else d - 0.5
    ax.text(x, i, f"{d:+.1f} pp", va="center", ha=ha,
            color=TEXT, fontsize=11, fontweight="bold")

for s in ax.spines.values():
    s.set_visible(False)

ax.invert_yaxis()
plt.tight_layout()
plt.savefig("success_delta_pp.png", transparent=True, bbox_inches="tight", pad_inches=0.15)
plt.close()

# =========================
# 3) RESTART BENCHMARK CARDS
# =========================
fig = plt.figure(figsize=(8.6, 4.8), dpi=300)
axes = [fig.add_axes(pos) for pos in [
    [0.03, 0.54, 0.45, 0.38],
    [0.52, 0.54, 0.45, 0.38],
    [0.03, 0.08, 0.45, 0.38],
    [0.52, 0.08, 0.45, 0.38],
]]

cards = [
    ("Corner",    blue["corner_pct"],    red["corner_pct"]),
    ("Throw-In",  blue["throwin_pct"],   red["throwin_pct"]),
    ("Cross",     blue["cross_pct"],     red["cross_pct"]),
    ("Free Kick", blue["freekick_pct"],  red["freekick_pct"]),
]

for ax, (label, b, r) in zip(axes, cards):
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 1)
    ax.axis("off")

    # Card background
    card = FancyBboxPatch(
        (0, 0), 1, 1,
        transform=ax.transAxes,
        boxstyle="round,pad=0.018,rounding_size=0.03",
        linewidth=1.2,
        edgecolor=CARD_EDGE,
        facecolor=CARD_BG
    )
    ax.add_patch(card)

    delta_val = b - r
    delta_color = GOOD if delta_val > 3 else BAD if delta_val < -3 else NEUTRAL

    # Header
    ax.text(0.05, 0.82, label, transform=ax.transAxes,
            color=TEXT, fontsize=12, fontweight="bold", ha="left")
    ax.text(0.95, 0.82, f"{delta_val:+.1f} pp", transform=ax.transAxes,
            color=delta_color, fontsize=11, fontweight="bold", ha="right")

    # Main number
    ax.text(0.05, 0.56, f"{b:.1f}%", transform=ax.transAxes,
            color=BLUE, fontsize=24, fontweight="bold", ha="left")

    # Progress track
    ax.add_patch(Rectangle((0.05, 0.26), 0.90, 0.10, transform=ax.transAxes,
                           facecolor=TRACK, edgecolor="none", alpha=0.7))
    ax.add_patch(Rectangle((0.05, 0.26), 0.90 * (b / 100), 0.10, transform=ax.transAxes,
                           facecolor=BLUE, edgecolor="none"))

    # Red benchmark marker
    benchmark_x = 0.05 + 0.90 * (r / 100)
    ax.add_line(plt.Line2D([benchmark_x, benchmark_x], [0.22, 0.40],
                           transform=ax.transAxes, color=RED, linewidth=2.2))

    # Footer text
    ax.text(0.05, 0.10, f"Red benchmark: {r:.1f}%", transform=ax.transAxes,
            color=MUTED, fontsize=10, ha="left")

plt.savefig("restart_benchmark_cards.png", transparent=True, bbox_inches="tight", pad_inches=0.12)
plt.close()

print("Saved: shot_profile_bars.png, success_delta_pp.png, restart_benchmark_cards.png")

Saved: shot_profile_bars.png, success_delta_pp.png, restart_benchmark_cards.png


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Wedge

# =========================
# DATA
# =========================

blue = {
    "shot_on_pct": 8 / 12 * 100,
    "shot_on_count": 8,
    "shot_total": 12,

    "pass_pct": 147 / 215 * 100,
    "dribble_pct": 10 / 14 * 100,
    "tackle_pct": 21 / 21 * 100,
    "interception_pct": 24 / 25 * 100,
}

red = {
    "shot_on_pct": 5 / 13 * 100,
    "shot_on_count": 5,
    "shot_total": 13,

    "pass_pct": 207 / 301 * 100,
    "dribble_pct": 14 / 22 * 100,
    "tackle_pct": 17 / 19 * 100,
    "interception_pct": 22 / 24 * 100,
}

# =========================
# STYLE
# =========================

BG = "#071221"          # outer page background if needed
CARD = "#0D1B2E"        # card background
CARD_EDGE = "#213652"
TEXT = "#EAF0FA"
MUTED = "#A6B5CC"
TRACK = "#334760"

BLUE = "#59A8FF"
BLUE_SOFT = "#A9D0FF"
RED = "#FF7A7A"
RED_SOFT = "#FFC2C2"

GOOD = "#32D7A6"
BAD = "#FF7A7A"
NEUTRAL = "#F6C85F"

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["figure.facecolor"] = "none"
plt.rcParams["axes.facecolor"] = "none"

# =========================
# HELPERS
# =========================

def pct_to_angle(pct):
    """Convert percentage to matplotlib pie angle with 90deg at top, clockwise."""
    return 90 - (pct / 100) * 360

def draw_donut(ax, pct, color, team_label, made, total):
    ax.set_aspect("equal")
    ax.axis("off")

    # Donut
    ax.pie(
        [pct, 100 - pct],
        startangle=90,
        counterclock=False,
        colors=[color, TRACK],
        wedgeprops=dict(width=0.30, edgecolor="none")
    )

    # Team label above donut
    ax.text(
        0, 1.18, team_label,
        ha="center", va="center",
        color=TEXT, fontsize=14, fontweight="bold"
    )

    # Minimal center text
    ax.text(
        0, 0.05, f"{pct:.1f}%",
        ha="center", va="center",
        color=TEXT, fontsize=24, fontweight="bold"
    )
    ax.text(
        0, -0.22, "On target",
        ha="center", va="center",
        color=MUTED, fontsize=11
    )

    # Caption below donut
    ax.text(
        0, -1.12, f"{made} of {total} shots",
        ha="center", va="center",
        color=MUTED, fontsize=11
    )

def draw_radial_tile(fig, box, title, blue_pct, red_pct):
    """
    Draw one radial benchmark tile inside figure coordinates.
    box = [x, y, w, h] in figure coords
    """
    ax = fig.add_axes(box)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    # Card
    card = FancyBboxPatch(
        (0, 0), 1, 1,
        boxstyle="round,pad=0.015,rounding_size=0.04",
        facecolor=CARD,
        edgecolor=CARD_EDGE,
        linewidth=1.2
    )
    ax.add_patch(card)

    # Title
    ax.text(
        0.08, 0.88, title,
        color=TEXT, fontsize=12, fontweight="bold",
        ha="left", va="center"
    )

    # Ring area as inset axes
    ring_ax = fig.add_axes([
        box[0] + box[2] * 0.08,
        box[1] + box[3] * 0.20,
        box[2] * 0.42,
        box[3] * 0.56
    ])
    ring_ax.set_aspect("equal")
    ring_ax.axis("off")

    # Blue ring progress
    ring_ax.pie(
        [blue_pct, 100 - blue_pct],
        startangle=90,
        counterclock=False,
        colors=[BLUE, TRACK],
        wedgeprops=dict(width=0.24, edgecolor="none")
    )

    # Red benchmark marker
    angle_deg = pct_to_angle(red_pct)
    angle_rad = np.deg2rad(angle_deg)

    r1, r2 = 0.88, 1.08
    x1, y1 = r1 * np.cos(angle_rad), r1 * np.sin(angle_rad)
    x2, y2 = r2 * np.cos(angle_rad), r2 * np.sin(angle_rad)
    ring_ax.plot([x1, x2], [y1, y2], color=RED, linewidth=3, solid_capstyle="round")

    # Main percentage
    ax.text(
        0.60, 0.58, f"{blue_pct:.1f}%",
        color=BLUE, fontsize=24, fontweight="bold",
        ha="left", va="center"
    )

    # Benchmark and delta
    delta = blue_pct - red_pct
    if delta > 3:
        delta_color = GOOD
    elif delta < -3:
        delta_color = BAD
    else:
        delta_color = NEUTRAL

    ax.text(
        0.60, 0.38, f"Red: {red_pct:.1f}%",
        color=MUTED, fontsize=11,
        ha="left", va="center"
    )
    ax.text(
        0.60, 0.22, f"{delta:+.1f} pp",
        color=delta_color, fontsize=12, fontweight="bold",
        ha="left", va="center"
    )

# =========================
# 1) SHOT DONUT COMPARISON
# =========================

fig = plt.figure(figsize=(8.8, 4.6), dpi=300)
fig.patch.set_alpha(0)

# Two donut axes
ax_left = fig.add_axes([0.05, 0.15, 0.36, 0.72])
ax_right = fig.add_axes([0.59, 0.15, 0.36, 0.72])

draw_donut(
    ax_left,
    blue["shot_on_pct"],
    BLUE,
    "Blue",
    blue["shot_on_count"],
    blue["shot_total"]
)

draw_donut(
    ax_right,
    red["shot_on_pct"],
    RED,
    "Red",
    red["shot_on_count"],
    red["shot_total"]
)

# Middle comparison note
delta_shot = blue["shot_on_pct"] - red["shot_on_pct"]
fig.text(
    0.50, 0.52, f"+{delta_shot:.1f} pp",
    ha="center", va="center",
    color=GOOD, fontsize=18, fontweight="bold"
)
fig.text(
    0.50, 0.46, "Blue edge",
    ha="center", va="center",
    color=MUTED, fontsize=10
)

plt.savefig(
    "shot_donut_comparison.png",
    dpi=300,
    transparent=True,
    bbox_inches="tight",
    pad_inches=0.15
)
plt.close()

# =========================
# 2) RADIAL BENCHMARK TILES
# =========================

fig = plt.figure(figsize=(10, 5.6), dpi=300)
fig.patch.set_alpha(0)

tiles = [
    ("Pass completion", blue["pass_pct"], red["pass_pct"]),
    ("Dribble success", blue["dribble_pct"], red["dribble_pct"]),
    ("Tackle success", blue["tackle_pct"], red["tackle_pct"]),
    ("Interception retention", blue["interception_pct"], red["interception_pct"]),
]

positions = [
    [0.04, 0.54, 0.44, 0.38],
    [0.52, 0.54, 0.44, 0.38],
    [0.04, 0.08, 0.44, 0.38],
    [0.52, 0.08, 0.44, 0.38],
]

for pos, tile in zip(positions, tiles):
    draw_radial_tile(fig, pos, tile[0], tile[1], tile[2])

plt.savefig(
    "radial_benchmark_tiles.png",
    dpi=300,
    transparent=True,
    bbox_inches="tight",
    pad_inches=0.12
)
plt.close()

print("Saved:")
print(" - shot_donut_comparison.png")
print(" - radial_benchmark_tiles.png")

Saved:
 - shot_donut_comparison.png
 - radial_benchmark_tiles.png


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# =========================
# DATA
# =========================
metrics = ["Pass completion", "Dribble success", "Tackle success", "Interception retention"]

blue = np.array([
    147/215 * 100,
    10/14 * 100,
    21/21 * 100,
    24/25 * 100
])

red = np.array([
    207/301 * 100,
    14/22 * 100,
    17/19 * 100,
    22/24 * 100
])

# Sort by Blue-Red difference if you want a cleaner story
delta = blue - red
order = np.argsort(delta)
metrics = [metrics[i] for i in order]
blue = blue[order]
red = red[order]
delta = delta[order]

# =========================
# STYLE
# =========================
TEXT = "#EAF0FA"
MUTED = "#A6B5CC"
GRID = "#2A4362"
BLUE = "#59A8FF"
BLUE_SOFT = "#A9D0FF"
RED = "#FF7A7A"
RED_SOFT = "#FFC2C2"
NEUTRAL = "#6E8099"

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.facecolor"] = "none"
plt.rcParams["figure.facecolor"] = "none"

# =========================
# FIGURE
# =========================
fig, ax = plt.subplots(figsize=(7.6, 5.0), dpi=300)

x_left = 0
x_right = 1
y = np.arange(len(metrics))

# horizontal guide lines
for yi in y:
    ax.hlines(yi, x_left, x_right, color=GRID, linestyle="--", lw=0.8, alpha=0.22, zorder=0)

# draw slopes
for i in range(len(metrics)):
    line_color = BLUE if delta[i] > 1 else RED if delta[i] < -1 else NEUTRAL
    ax.plot([x_left, x_right], [red[i], blue[i]], color=line_color, lw=2.6, alpha=0.95, zorder=2)

    # endpoints
    ax.scatter(x_left, red[i], s=90, color=RED, zorder=3)
    ax.scatter(x_right, blue[i], s=90, color=BLUE, zorder=3)

    # metric label in the middle-left region
    ax.text(
        0.5, (red[i] + blue[i]) / 2,
        metrics[i],
        ha="center", va="center",
        color=TEXT, fontsize=11, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.22", facecolor="none", edgecolor="none")
    )

    # endpoint labels
    ax.text(
        x_left - 0.03, red[i],
        f"{red[i]:.1f}%",
        ha="right", va="center",
        color=RED_SOFT, fontsize=10
    )
    ax.text(
        x_right + 0.03, blue[i],
        f"{blue[i]:.1f}%",
        ha="left", va="center",
        color=BLUE_SOFT, fontsize=10
    )

# side labels
ax.text(x_left, 103, "Red", ha="center", va="bottom", color=RED_SOFT, fontsize=12, fontweight="bold")
ax.text(x_right, 103, "Blue", ha="center", va="bottom", color=BLUE_SOFT, fontsize=12, fontweight="bold")

# axis formatting
ax.set_xlim(-0.12, 1.12)
ax.set_ylim(55, 105)
ax.set_xticks([])
ax.set_yticks([])

for s in ax.spines.values():
    s.set_visible(False)

plt.tight_layout()
plt.savefig("execution_edge_slope.png", transparent=True, bbox_inches="tight", pad_inches=0.15)
plt.close()

print("Saved: execution_edge_slope.png")

Saved: execution_edge_slope.png


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle, Circle

# =========================
# DATA
# =========================
raw = {
    "Pass":         {"blue_success": 147, "blue_fail": 68, "red_success": 207, "red_fail": 94},
    "Dribble":      {"blue_success": 10,  "blue_fail": 4,  "red_success": 14,  "red_fail": 8},
    "Tackle":       {"blue_success": 21,  "blue_fail": 0,  "red_success": 17,  "red_fail": 2},
    "Interception": {"blue_success": 24,  "blue_fail": 1,  "red_success": 22,  "red_fail": 2},
}

df = pd.DataFrame(raw).T.reset_index().rename(columns={"index": "Metric"})
for team in ["blue", "red"]:
    df[f"{team}_total"] = df[f"{team}_success"] + df[f"{team}_fail"]
    df[f"{team}_rate"] = df[f"{team}_success"] / df[f"{team}_total"] * 100
df["delta_pp"] = df["blue_rate"] - df["red_rate"]

# Keep order
metric_order = ["Pass", "Dribble", "Tackle", "Interception"]
df["Metric"] = pd.Categorical(df["Metric"], categories=metric_order, ordered=True)
df = df.sort_values("Metric").reset_index(drop=True)

# =========================
# STYLE
# =========================
TEXT = "#EAF0FA"
MUTED = "#A6B5CC"
GRID = "#2A4362"
BLUE = "#59A8FF"
BLUE_SOFT = "#A9D0FF"
RED = "#FF7A7A"
RED_SOFT = "#FFC2C2"
TRACK = "#334760"
CARD_BG = "#0D1B2E"
CARD_EDGE = "#213652"
GOOD = "#32D7A6"
BAD = "#FF7A7A"
NEUTRAL = "#F6C85F"

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.facecolor"] = "none"
plt.rcParams["figure.facecolor"] = "none"

# =========================
# HELPERS
# =========================
def delta_color(delta):
    if delta > 2:
        return GOOD
    elif delta < -2:
        return BAD
    return NEUTRAL

def add_card(ax, radius=0.04):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    card = FancyBboxPatch(
        (0, 0), 1, 1,
        boxstyle=f"round,pad=0.015,rounding_size={radius}",
        linewidth=1.2,
        edgecolor=CARD_EDGE,
        facecolor=CARD_BG
    )
    ax.add_patch(card)

def draw_waffle(ax, x0, y0, width, height, pct, fill_color, empty_color=TRACK, rows=10, cols=10):
    """Draws a 10x10 waffle in axes coords."""
    filled = int(round(pct))
    gap = 0.005
    cell_w = (width - gap * (cols - 1)) / cols
    cell_h = (height - gap * (rows - 1)) / rows

    count = 0
    for r in range(rows):
        for c in range(cols):
            # fill from bottom-left
            rr = rows - 1 - r
            x = x0 + c * (cell_w + gap)
            y = y0 + rr * (cell_h + gap)
            color = fill_color if count < filled else empty_color
            rect = Rectangle((x, y), cell_w, cell_h, transform=ax.transAxes,
                             facecolor=color, edgecolor="none")
            ax.add_patch(rect)
            count += 1

# =========================
# OPTION 1: BUBBLE MATRIX
# =========================
def save_option_1_bubble_matrix(df, filename="option1_bubble_matrix.png"):
    fig, ax = plt.subplots(figsize=(8.4, 5.2), dpi=300)

    # Bubble size uses sqrt scaling so passes don't dominate too hard
    max_total = df[["blue_total", "red_total"]].to_numpy().max()

    y_positions = np.arange(len(df))[::-1]
    ax.set_xlim(0, 100)
    ax.set_ylim(-0.6, len(df)-0.4)

    # grid lines
    for x in range(0, 101, 20):
        ax.axvline(x, color=GRID, lw=0.8, ls="--", alpha=0.25, zorder=0)
    for y in y_positions:
        ax.hlines(y, 0, 100, color=GRID, lw=0.8, alpha=0.15, zorder=0)

    for i, row in df.iterrows():
        y = y_positions[i]

        b_size = 250 + 1800 * math.sqrt(row["blue_total"] / max_total)
        r_size = 250 + 1800 * math.sqrt(row["red_total"] / max_total)

        # team markers
        ax.scatter(row["red_rate"], y, s=r_size, color=RED, alpha=0.82, zorder=3)
        ax.scatter(row["blue_rate"], y, s=b_size, color=BLUE, alpha=0.90, zorder=4)

        # metric label
        ax.text(-4, y, row["Metric"], ha="right", va="center",
                color=TEXT, fontsize=12, fontweight="bold")

        # percentage labels
        ax.text(row["red_rate"], y + 0.26, f'{row["red_rate"]:.1f}%',
                ha="center", va="bottom", color=RED_SOFT, fontsize=9)
        ax.text(row["blue_rate"], y - 0.28, f'{row["blue_rate"]:.1f}%',
                ha="center", va="top", color=BLUE_SOFT, fontsize=9)

        # attempt labels
        ax.text(row["red_rate"], y, f'{int(row["red_total"])}',
                ha="center", va="center", color=TEXT, fontsize=10, fontweight="bold")
        ax.text(row["blue_rate"], y, f'{int(row["blue_total"])}',
                ha="center", va="center", color=TEXT, fontsize=10, fontweight="bold")

    # top labels
    ax.text(0, len(df)-0.15, "0%", color=MUTED, fontsize=9, ha="left")
    ax.text(100, len(df)-0.15, "100%", color=MUTED, fontsize=9, ha="right")
    ax.text(50, len(df)-0.15, "Success rate →", color=MUTED, fontsize=10, ha="center")
    ax.text(90, -0.95, "Bubble size = total attempts", color=MUTED, fontsize=9, ha="right")
    ax.text(4, -0.95, "Red", color=RED_SOFT, fontsize=10, fontweight="bold", ha="left")
    ax.text(14, -0.95, "Blue", color=BLUE_SOFT, fontsize=10, fontweight="bold", ha="left")

    ax.set_xticks([])
    ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)

    plt.tight_layout()
    plt.savefig(filename, transparent=True, bbox_inches="tight", pad_inches=0.15)
    plt.close()

# =========================
# OPTION 2: DUAL WAFFLE CARDS
# =========================
def save_option_2_dual_waffle_cards(df, filename="option2_dual_waffle_cards.png"):
    fig = plt.figure(figsize=(10, 6), dpi=300)
    fig.patch.set_alpha(0)

    positions = [
        [0.04, 0.54, 0.44, 0.38],
        [0.52, 0.54, 0.44, 0.38],
        [0.04, 0.08, 0.44, 0.38],
        [0.52, 0.08, 0.44, 0.38],
    ]

    for pos, (_, row) in zip(positions, df.iterrows()):
        ax = fig.add_axes(pos)
        add_card(ax)

        # title
        ax.text(0.06, 0.88, row["Metric"], color=TEXT, fontsize=12, fontweight="bold", ha="left")
        ax.text(0.94, 0.88, f'{row["delta_pp"]:+.1f} pp',
                color=delta_color(row["delta_pp"]), fontsize=11, fontweight="bold", ha="right")

        # Blue / Red labels
        ax.text(0.22, 0.77, "Blue", color=BLUE_SOFT, fontsize=10, fontweight="bold", ha="center")
        ax.text(0.74, 0.77, "Red", color=RED_SOFT, fontsize=10, fontweight="bold", ha="center")

        # Waffles
        draw_waffle(ax, x0=0.06, y0=0.22, width=0.32, height=0.45, pct=row["blue_rate"], fill_color=BLUE)
        draw_waffle(ax, x0=0.58, y0=0.22, width=0.32, height=0.45, pct=row["red_rate"], fill_color=RED)

        # % labels
        ax.text(0.22, 0.13, f'{row["blue_rate"]:.1f}%', color=TEXT, fontsize=16, fontweight="bold", ha="center")
        ax.text(0.74, 0.13, f'{row["red_rate"]:.1f}%', color=TEXT, fontsize=16, fontweight="bold", ha="center")

        # count labels
        ax.text(0.22, 0.06, f'{int(row["blue_success"])}/{int(row["blue_total"])}',
                color=MUTED, fontsize=10, ha="center")
        ax.text(0.74, 0.06, f'{int(row["red_success"])}/{int(row["red_total"])}',
                color=MUTED, fontsize=10, ha="center")

    plt.savefig(filename, transparent=True, bbox_inches="tight", pad_inches=0.12)
    plt.close()

# =========================
# OPTION 3: SCORE TILES
# =========================
def save_option_3_score_tiles(df, filename="option3_score_tiles.png"):
    fig = plt.figure(figsize=(10, 6), dpi=300)
    fig.patch.set_alpha(0)

    positions = [
        [0.04, 0.54, 0.44, 0.38],
        [0.52, 0.54, 0.44, 0.38],
        [0.04, 0.08, 0.44, 0.38],
        [0.52, 0.08, 0.44, 0.38],
    ]

    for pos, (_, row) in zip(positions, df.iterrows()):
        ax = fig.add_axes(pos)
        add_card(ax)

        # header
        ax.text(0.06, 0.86, row["Metric"], color=TEXT, fontsize=12, fontweight="bold", ha="left")
        ax.text(0.94, 0.86, f'{row["delta_pp"]:+.1f} pp',
                color=delta_color(row["delta_pp"]), fontsize=11, fontweight="bold", ha="right")

        # center divider
        ax.plot([0.50, 0.50], [0.18, 0.74], color=CARD_EDGE, lw=1.2)

        # Blue block
        ax.text(0.25, 0.70, "Blue", color=BLUE_SOFT, fontsize=10, fontweight="bold", ha="center")
        ax.text(0.25, 0.49, f'{row["blue_rate"]:.1f}%', color=BLUE, fontsize=24, fontweight="bold", ha="center")
        ax.text(0.25, 0.28, f'{int(row["blue_success"])}/{int(row["blue_total"])}',
                color=MUTED, fontsize=11, ha="center")
        ax.text(0.25, 0.13, "successful / total", color=MUTED, fontsize=9, ha="center")

        # Red block
        ax.text(0.75, 0.70, "Red", color=RED_SOFT, fontsize=10, fontweight="bold", ha="center")
        ax.text(0.75, 0.49, f'{row["red_rate"]:.1f}%', color=RED, fontsize=24, fontweight="bold", ha="center")
        ax.text(0.75, 0.28, f'{int(row["red_success"])}/{int(row["red_total"])}',
                color=MUTED, fontsize=11, ha="center")
        ax.text(0.75, 0.13, "successful / total", color=MUTED, fontsize=9, ha="center")

        # small center badge
        badge = Circle((0.50, 0.49), 0.06, transform=ax.transAxes,
                       facecolor="#12243b", edgecolor=CARD_EDGE, linewidth=1.0)
        ax.add_patch(badge)
        ax.text(0.50, 0.49, "vs", color=MUTED, fontsize=9, fontweight="bold", ha="center", va="center")

    plt.savefig(filename, transparent=True, bbox_inches="tight", pad_inches=0.12)
    plt.close()

# =========================
# RUN ALL
# =========================
save_option_1_bubble_matrix(df)
save_option_2_dual_waffle_cards(df)
save_option_3_score_tiles(df)

print("Saved:")
print(" - option1_bubble_matrix.png")
print(" - option2_dual_waffle_cards.png")
print(" - option3_score_tiles.png")

Saved:
 - option1_bubble_matrix.png
 - option2_dual_waffle_cards.png
 - option3_score_tiles.png
